# LightGBM

Nesse notebook nós iremos treinar um modelo LightGBM em nosso dataset médico. O objetivo aqui é checar possíveis melhorias usando a abordagem de um modelo baseado árvores de decisão e boosting.

## Importando as Bibliotecas

Primeiro vamos carregar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carregando o Dataset

Agora vamos carregar nosso dataset.

In [2]:
df = pd.read_parquet("../../data/processed/UCMF_fitted.parquet")

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 11705 entries, 0 to 12872
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   peso                    9587 non-null   Float64 
 1   altura                  8078 non-null   Int64   
 2   imc                     7708 non-null   Int64   
 3   idade                   10865 non-null  Float64 
 4   pulsos                  11657 non-null  category
 5   pa_sistolica            5131 non-null   Int64   
 6   pa_diastolica           5121 non-null   Int64   
 7   ppa                     10768 non-null  category
 8   patologia               11705 non-null  category
 9   b2                      11674 non-null  category
 10  sopro                   11682 non-null  category
 11  fc                      10986 non-null  Int64   
 12  hda1                    8565 non-null   category
 13  hda2                    11705 non-null  category
 14  sexo                    11701 non-null

## Seleção de Modelo

Assim como fizemos com o `XGBoost`, vamos usar o `Optuna` para otimizar os hiperparametros do `LightGBM`.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import (
    SimpleImputer,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)

def create_preprocessor(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough"
    )

    return preprocessor

Vamos criar nossa função objetivo.

In [7]:
from lightgbm import LGBMClassifier

random_state = 42

cross_validator = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

def create_objetive(X, y):
    preprocessor = create_preprocessor(X)

    def objective(trial):
        model = LGBMClassifier(
            objective="binary",
            random_state=random_state,
            class_weight="balanced",
            n_estimators=trial.suggest_int("n_estimators", 200, 1000),
            learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 16, 64),
            max_depth=trial.suggest_int("max_depth", 3, 12),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 0.000001, 10, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 0.000001, 10, log=True),
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cross_validator,
            scoring="roc_auc",
        )

        return scores.mean()
    
    return objective

Então vamos criar nossa função objetivo com o dataset inteiro.

In [8]:
label = LabelEncoder()

X = df.drop(["patologia"], axis="columns")
y = label.fit_transform(df["patologia"])

objective = create_objetive(X, y)

E agora vamos otimizar os hiperparâmetros.

In [ ]:
import optuna

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.PercentilePruner(25.0, n_startup_trials=5)
)

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True,
)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-13 17:09:55,931] A new study created in memory with name: no-name-37317380-69cb-4c9b-ac08-3f563ed99155
  0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 3968, number of negative: 5396
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002900 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1277
[LightGBM] [Info] Number of data points in the train set: 9364, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 3968, number of negative: 5396
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002172 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1277
[LightGBM] [Info] Number of data points i

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 3968, number of negative: 5396
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004749 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1280
[LightGBM] [Info] Number of data points in the train set: 9364, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: 

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 7. Best value: 0.948064:  80%|████████  | 8/10 [00:56<00:11,  5.93s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[I 2026-06-13 17:10:52,422] Trial 7 finished with value: 0.9480644264568736 and parameters: {'n_estimators': 372, 'learning_rate': 0.04640338455884896, 'num_leaves': 50, 'max_depth': 10, 'min_child_samples': 61, 'subsample': 0.5794928371140281, 'colsample_bytree': 0.6491805290874102, 'reg_alpha': 4.589492619329257e-05, 'reg_lambda': 0.01703249996052547}. Best is trial 7 with value: 0.9480644264568736.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 7. Best value: 0.948064:  90%|█████████ | 9/10 [00:57<00:06,  6.37s/it]


[I 2026-06-13 17:10:53,247] Trial 5 finished with value: 0.9468689471292953 and parameters: {'n_estimators': 903, 'learning_rate': 0.09309353238501632, 'num_leaves': 53, 'max_depth': 9, 'min_child_samples': 10, 'subsample': 0.997895598637203, 'colsample_bytree': 0.8288596415326618, 'reg_alpha': 0.7299745089706243, 'reg_lambda': 2.296135905396399e-06}. Best is trial 7 with value: 0.9480644264568736.


KeyboardInterrupt: 

Agora vamos montar o pipeline do nosso modelo ótimo.

In [14]:
params = study.best_params

model = XGBClassifier(
    **params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=random_state,
    tree_method="hist",
)

preprocessor = create_preprocessor(X)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

E vamos avaliar ele.

In [17]:
scores = cross_validate(
    pipeline,
    X,
    y,
    cv=cross_validator,
    scoring={
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    n_jobs=-1,
)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [27]:
for metric, values in scores.items():
    print(f"{metric:14} = {(100*values.mean()).round(2)}")

fit_time       = 1067.33
score_time     = 45.01
test_roc_auc   = 95.24
test_accuracy  = 92.87
test_precision = 95.05
test_recall    = 87.76
test_f1        = 91.26


Como podemos notar, as métricas são tão excelentes quanto nos modelos anteriores. Vamos então analisar a importância das nossas features.

In [38]:
pipeline.fit(X, y)

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
feature_importances = pipeline.named_steps["model"].feature_importances_

importances = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances,
}).sort_values(by="importance", ascending=False)

print(importances.to_string())

                                                          feature  importance
31                          categorical_features__sopro_sistolico    0.657263
26                                categorical_features__b2_normal    0.032950
52             categorical_features__motivo1_parecer_cardiologico    0.021862
25                         categorical_features__b2_hiperfonetica    0.016681
68                            categorical_features__motivo2_sopro    0.016661
57            categorical_features__motivo2_cardiopatia_congenica    0.016121
29                           categorical_features__sopro_continuo    0.010034
16                           categorical_features__pulsos_normais    0.006807
28                                 categorical_features__b2_unica    0.006796
27                                 categorical_features__b2_outro    0.006104
30                         categorical_features__sopro_diastolico    0.006033
50                         categorical_features__motivo1_check_u

Como podemos notar, a feature mais importante continua sendo `"sopro"` com valor `"sistolico"` e ela é disparada a feature mais importante.

Agora, diferente do que fizemos nos modelos anteriores, vamos refazer a otimização de hiperparametros mas sem a variável `"sopro"`. A motivação para isso é o fato de que modelos baseados em Boosting com o XGBoost, LightGBM e CatBoost são muito poderoso e talvez eles encontrem alguma relação entre as features sem `"sopro"` e o target que os outros modelos não encontraram.

In [29]:
objective_sem_sopro = create_objetive(X.drop("sopro", axis="columns"), y)

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.PercentilePruner(25.0, n_startup_trials=5)
)

study.optimize(
    objective_sem_sopro,
    n_trials=100,
    show_progress_bar=True,
    n_jobs=-1
)

[I 2026-06-13 16:28:41,681] A new study created in memory with name: no-name-4c54104c-f463-4e00-b1ee-3df3a9c07f20
  0%|          | 0/100 [00:00<?, ?it/s]/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 2. Best value: 0.808122:   1%|          | 1/100 [00:04<07:17,  4.41s/it]

[I 2026-06-13 16:28:46,089] Trial 2 finished with value: 0.808121756856931 and parameters: {'n_estimators': 947, 'learning_rate': 0.05133564064399223, 'max_depth': 6, 'min_child_weight': 17, 'subsample': 0.6726689330451595, 'colsample_bytree': 0.9982749261336382, 'gamma': 7.655383897216025, 'reg_alpha': 0.013366866159396874, 'reg_lambda': 1.1689394382473777e-05, 'scale_pos_weight': 1.576491964116663}. Best is trial 2 with value: 0.808121756856931.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 3. Best value: 0.808395:   2%|▏         | 2/100 [00:10<08:49,  5.40s/it]

[I 2026-06-13 16:28:52,180] Trial 3 finished with value: 0.8083945096726369 and parameters: {'n_estimators': 1959, 'learning_rate': 0.09285514312573959, 'max_depth': 7, 'min_child_weight': 14, 'subsample': 0.8249645473219476, 'colsample_bytree': 0.8652697575559887, 'gamma': 9.358653112961772, 'reg_alpha': 0.004081113558767837, 'reg_lambda': 0.0041138096947447396, 'scale_pos_weight': 1.6921588644376313}. Best is trial 3 with value: 0.8083945096726369.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 3. Best value: 0.808395:   3%|▎         | 3/100 [00:13<07:09,  4.43s/it]

...

[I 2026-06-13 16:33:04,095] Trial 97 finished with value: 0.8088693237523614 and parameters: {'n_estimators': 1750, 'learning_rate': 0.06290577503452827, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8158152140226516, 'colsample_bytree': 0.6196191502028081, 'gamma': 4.279630605017196, 'reg_alpha': 0.008835339451940219, 'reg_lambda': 0.5355643250865622, 'scale_pos_weight': 1.9637189113752074}. Best is trial 57 with value: 0.8113764078528899.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 57. Best value: 0.811376:  99%|█████████▉| 99/100 [04:26<00:03,  3.14s/it]

[I 2026-06-13 16:33:08,173] Trial 98 finished with value: 0.8110594167722806 and parameters: {'n_estimators': 1740, 'learning_rate': 0.006954407666796061, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8213502410315787, 'colsample_bytree': 0.6514432913030896, 'gamma': 4.319998651952965, 'reg_alpha': 0.007730964377156366, 'reg_lambda': 1.1099789506227005, 'scale_pos_weight': 1.9717956749950338}. Best is trial 57 with value: 0.8113764078528899.


Best trial: 57. Best value: 0.811376: 100%|██████████| 100/100 [04:28<00:00,  2.68s/it]

[I 2026-06-13 16:33:09,707] Trial 99 finished with value: 0.8046562268346923 and parameters: {'n_estimators': 1759, 'learning_rate': 0.1631254591009625, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8919106141407036, 'colsample_bytree': 0.5883580155966659, 'gamma': 3.2602614891061723, 'reg_alpha': 0.008423678247301981, 'reg_lambda': 1.184994289341145, 'scale_pos_weight': 2.1523341327477286}. Best is trial 57 with value: 0.8113764078528899.


Vamos montar nosso pipeline.

In [ ]:
params_sem_sopro = study.best_params

model_sem_sopro = XGBClassifier(
    **params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=random_state,
    tree_method="hist",
)

preprocessor_sem_sopro = create_preprocessor(X.drop("sopro", axis="columns"))

pipeline_sem_sopro = Pipeline(steps=[
    ("preprocessor", preprocessor_sem_sopro),
    ("model", model_sem_sopro)
])

E calcular as métricas de desempenho.

In [33]:
scores_sem_sopro = cross_validate(
    pipeline_sem_sopro,
    X.drop("sopro", axis="columns"),
    y,
    cv=cross_validator,
    scoring={
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    n_jobs=-1,
)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [34]:
for metric, values in scores_sem_sopro.items():
    print(f"{metric:14} = {(100*values.mean()).round(2)}")

fit_time       = 1166.9
score_time     = 43.62
test_roc_auc   = 80.9
test_accuracy  = 73.79
test_precision = 68.46
test_recall    = 70.77
test_f1        = 69.59


Como podemos ver, as métricas sem a feature `"sopro"` continuam baixas como nos demais modelos, mas tivemos uma melhoria no recall.